In [ ]:
import os
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision.ops import nms

import SimpleITK as sitk
import numpy as np 

import plotly.graph_objects as go
from plotly.subplots import make_subplots


from lightning import LightningDataModule
import monai
import random 
import albumentations as A

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap, BoundaryNorm
import pickle


In [ ]:
def _resize_with_grid_sample(
    x: torch.Tensor,
    out_hw: tuple[int, int],
    mode: str,
    align_corners: bool = False,
) -> torch.Tensor:
    """
    x: (N,C,H,W) tensor
    out_hw: (H_out, W_out)
    mode: 'bilinear' for images/logits, 'nearest' for label maps
    """
    assert x.ndim == 4, f"Expected (N,C,H,W), got {x.shape}"

    mesh_grid_params = [torch.arange(start=-1.0, end=1.0, step=(2.0/s), device=x.device) for s in out_hw]
    mesh_grid = torch.stack(torch.meshgrid(mesh_grid_params, indexing='xy'), dim=-1).to(torch.float32).unsqueeze(0)

    y = F.grid_sample(
        x, mesh_grid,
        mode=mode,
        padding_mode="zeros",
        align_corners=align_corners,
    )
    return y

@torch.no_grad()
def run_segmentation_resize_roundtrip(
    model,
    img: torch.Tensor,
    *,
    in_hw: tuple[int, int] = (512, 512),
    align_corners: bool = False,
    return_logits: bool = True,
):
    """
    img: (N,C,H,W)
    Returns:
      - logits_back: (N,K,H,W) if return_logits=True
      - or labels_back: (N,1,H,W) long tensor if return_logits=False
    """
    assert img.ndim == 4, f"Expected (N,C,H,W), got {img.shape}"
    n, c, h0, w0 = img.shape

    # 1) resize to model input
    img_512 = _resize_with_grid_sample(
        img, in_hw, mode="nearest", align_corners=align_corners
    )

    # 2) run model (expects 512x512)
    logits_512 = model(img_512)  # (N,K,512,512) typically

    # 3) resize back to original size
    logits_back = _resize_with_grid_sample(
        logits_512.float(), (w0, h0), mode="nearest", align_corners=align_corners
    )

    if return_logits:
        return logits_back

    # If you want discrete labels, do argmax at original resolution
    labels_back = torch.argmax(logits_back, dim=1, keepdim=True).to(torch.long)  # (N,1,H,W)
    return labels_back, img_512, logits_512


In [ ]:

def _make_label_cmap_and_norm(label_colors, background_label=0, default_color="black"):
    """
    Build a discrete ListedColormap + BoundaryNorm so integer labels map to fixed colors.
    """
    if not isinstance(label_colors, dict) or len(label_colors) == 0:
        raise ValueError("label_colors must be a non-empty dict like {0:'tab:blue', 1:'tab:orange', ...}")

    max_label = int(max(label_colors.keys()))
    colors = [label_colors.get(i, default_color) for i in range(max_label + 1)]

    cmap = ListedColormap(colors)

    # bins centered on integers: [-0.5, 0.5, 1.5, ...]
    boundaries = np.arange(-0.5, max_label + 1.5, 1.0)
    norm = BoundaryNorm(boundaries=boundaries, ncolors=len(colors), clip=False)

    return cmap, norm, max_label


def plot_image(
    imgs,
    labelmaps,
    idx=0,
    alpha=0.6,
    label_colors=None,
    background_label=0,
    title=None,
    figsize=(6, 6),
    show_colorbar=True,
    colorbar_labels=None,  # optional dict {label:int -> "name"}
):
    """
    Plot a single image with a discrete labelmap overlay using matplotlib.

    Args:
        imgs: torch.Tensor or np.ndarray of shape (B, C, H, W)
        labelmaps: torch.Tensor or np.ndarray of shape (B, H, W)
        idx: batch index to display
        alpha: transparency of labelmap overlay
        label_colors: dict mapping int label -> matplotlib color string
                      e.g. {0:"tab:blue", 1:"tab:orange", ...}
        background_label: label value to treat as background (hidden)
        title: optional plot title
        figsize: matplotlib figure size
        show_colorbar: whether to show a categorical colorbar
        colorbar_labels: optional dict mapping label -> display string for colorbar ticks
    """
    if label_colors is None:
        raise ValueError("Please pass label_colors dict, e.g. {0:'tab:blue', 1:'tab:orange', ...}")

    # --- move to numpy ---
    if hasattr(imgs, "detach"):
        img = imgs[idx].detach().cpu().float().numpy()
    else:
        img = np.asarray(imgs[idx], dtype=np.float32)

    if hasattr(labelmaps, "detach"):
        seg = labelmaps[idx].detach().cpu().numpy()
    else:
        seg = np.asarray(labelmaps[idx])

    # --- checks ---
    if img.ndim != 3:
        raise ValueError(f"Expected img (C,H,W), got {img.shape}")
    if seg.ndim != 2:
        raise ValueError(f"Expected labelmap (H,W), got {seg.shape}")

    C, H, W = img.shape
    if seg.shape != (H, W):
        raise ValueError(f"Labelmap shape {seg.shape} != image shape {(H, W)}")

    # --- prepare RGB image ---
    if C == 1:
        rgb = np.repeat(img[0:1], 3, axis=0)
    else:
        rgb = img[:3]

    # normalize to [0,1]
    vmin, vmax = np.min(rgb), np.max(rgb)
    if vmax > vmin:
        rgb = (rgb - vmin) / (vmax - vmin)
    else:
        rgb = np.zeros_like(rgb)

    rgb = np.transpose(rgb, (1, 2, 0))  # (H,W,3)

    # --- discrete colormap + norm ---
    cmap, norm, max_label = _make_label_cmap_and_norm(
        label_colors=label_colors,
        background_label=background_label,
    )

    # --- mask background labels (invisible) ---
    seg_vis = seg.astype(np.float32)
    seg_vis[seg_vis == background_label] = np.nan

    # --- plot ---
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb)
    im = ax.imshow(
        seg_vis,
        cmap=cmap,
        norm=norm,
        alpha=alpha,
        interpolation="nearest",
    )

    ax.axis("off")
    if title:
        ax.set_title(title)

    # --- optional categorical colorbar ---
    if show_colorbar:
        # show ticks for all labels except background
        labels_sorted = [l for l in range(max_label + 1) if l != background_label]
        cbar = plt.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04,
            ticks=labels_sorted,
        )
        if colorbar_labels is not None:
            cbar.ax.set_yticklabels([colorbar_labels.get(l, str(l)) for l in labels_sorted])
        else:
            cbar.ax.set_yticklabels([str(l) for l in labels_sorted])

    plt.tight_layout()
    plt.show()


In [ ]:
class TTDatasetBX(Dataset):
    def __init__(self, df, mount_point = "./", transform=None, img_column="img_path", class_column = 'class',sev_column='sev'):
        self.df = df
        self.mount_point = mount_point
        self.transform = transform
        self.img_column = img_column
        self.class_column = class_column
        self.severity_column = sev_column
        self.transform = transform

        self.df_subject = self.df[img_column].drop_duplicates().reset_index()
        self.target_size = (768, 1536)

    def __len__(self):
        return len(self.df_subject.index)

    def __getitem__(self, idx):
        
        subject = self.df_subject.iloc[idx][self.img_column]
        img_path = os.path.join(self.mount_point, subject)
        # self.seg_path = img_path.replace('img', 'segmentation_cleaned').replace('.jpg', '.nrrd')
        self.seg_path = img_path.replace('img', 'seg').replace('.jpg', '.nrrd')

        df_patches = self.df.loc[ self.df[self.img_column] == subject]        

        seg = torch.tensor(np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(self.seg_path)).copy())).to(torch.float32)
        img = torch.tensor(np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(img_path)).copy())).to(torch.float32)
        img = img.permute((2, 0, 1))
        img = img/255.0

        ## crop img within segmentation
        bbx_eye = self.compute_eye_bbx(seg, pad=0.05)
        img_cropped = img[:,bbx_eye[1]:bbx_eye[3],bbx_eye[0]:bbx_eye[2] ]
        seg_cropped = seg[bbx_eye[1]:bbx_eye[3],bbx_eye[0]:bbx_eye[2] ]
        # seg_cropped[ seg_cropped!=3 ] =0
        H, W = seg_cropped.shape
        
        self.pad = int(img_cropped.shape[1]/10)

        df_filtered = df_patches[(df_patches['x_patch'] >= bbx_eye[0].numpy()) & (df_patches['x_patch'] <= bbx_eye[2].numpy())]
        df_filtered = df_filtered[(df_filtered['y_patch'] >= bbx_eye[1].numpy()) & (df_filtered['y_patch'] <= bbx_eye[3].numpy())]

        bbx, classes = [], []
        if not df_filtered.empty:
            for k, row in df_filtered.iterrows():
                class_idx =  torch.tensor(row[self.class_column]).to(torch.long)
                x, y = row['x_patch'], row['y_patch']

                cropped_x, cropped_y = x - bbx_eye[0], y -bbx_eye[1]
                box = torch.tensor([max((cropped_x-2*self.pad/3), 0),
                                    max((cropped_y-5*self.pad/3), 0),
                                    min((cropped_x+2*self.pad/3), img_cropped.shape[2]),
                                    min((cropped_y+self.pad/3), img_cropped.shape[1])])

                classes.append(class_idx.unsqueeze(0))
                bbx.append(box.unsqueeze(0))

        else:
            classes.append(torch.tensor(1).to(torch.long).unsqueeze(0))
            bbx.append(torch.tensor([5,5,img_cropped.shape[2]-5, img_cropped.shape[1]-5]).unsqueeze(0))
        bbx, classes = torch.cat(bbx), torch.cat(classes)

        augmented = self.transform(img_cropped.permute(1,2,0).numpy(), bbx.numpy(), classes.numpy(), seg_cropped.numpy())

        aug_coords = torch.tensor(augmented['bboxes'])
        aug_image = augmented['image']
        aug_seg = augmented['mask']
        aug_image = torch.tensor(aug_image).permute(2,0,1)

        indices = nms(aug_coords, 0.5*torch.ones_like(aug_coords[:,0]), iou_threshold=1.0) ## iou as args
        return {"img": aug_image, 
                "labels": classes[indices], 
                "boxes": aug_coords[indices] ,
                'mask':torch.tensor(aug_seg), 
                }


    def compute_eye_bbx(self, seg, label=1, pad=0):

        shape = seg.shape
        
        ij = torch.argwhere(seg.squeeze() != 0)

        bb = torch.tensor([0, 0, 0, 0])# xmin, ymin, xmax, ymax

        bb[0] = torch.clip(torch.min(ij[:,1]) - shape[1]*pad, 0, shape[1])
        bb[1] = torch.clip(torch.min(ij[:,0]) - shape[0]*pad, 0, shape[0])
        bb[2] = torch.clip(torch.max(ij[:,1]) + shape[1]*pad, 0, shape[1])
        bb[3] = torch.clip(torch.max(ij[:,0]) + shape[0]*pad, 0, shape[0])
        
        return bb


    def get_xy_coordinates_from_patch_name(self,patch_name):
        for elt in patch_name.split('_'):
            if 'x' == elt[-1]:
                x = elt[:-1]
            elif elt == 'Wavy':
                pass
            elif 'y' == elt[-1]:
                y = elt[:-1]
        return int(x), int(y)

class TTDataModuleBX(LightningDataModule):
    def __init__(self, df_train, df_val, df_test, mount_point="./", batch_size=256, num_workers=4, img_column="img_path", class_column='class', severity_column='sev', balanced=False, train_transform=None, valid_transform=None, test_transform=None, drop_last=False):
        super().__init__()

        self.df_train = df_train
        self.df_val = df_val
        self.df_test = df_test

        self.mount_point = mount_point
        self.batch_size = batch_size
        self.num_workers = num_workers
        
        self.img_column = img_column
        self.class_column = class_column   
        self.severity_column = severity_column   
        
        self.balanced = balanced
        self.train_transform = train_transform
        self.valid_transform = valid_transform
        self.test_transform = test_transform
        self.drop_last=drop_last

    def setup(self, stage=None):

        # Assign train/val datasets for use in dataloaders
        self.train_ds = monai.data.Dataset(data=TTDatasetBX(self.df_train, mount_point=self.mount_point, img_column=self.img_column, class_column=self.class_column, sev_column=self.severity_column, transform=self.train_transform))
        self.val_ds = monai.data.Dataset(TTDatasetBX(self.df_val, mount_point=self.mount_point, img_column=self.img_column, class_column=self.class_column, sev_column=self.severity_column, transform=self.valid_transform))
        self.test_ds = monai.data.Dataset(TTDatasetBX(self.df_test, mount_point=self.mount_point, img_column=self.img_column, class_column=self.class_column, sev_column=self.severity_column, transform=self.test_transform))

    def train_dataloader(self):

        # if self.balanced: 
        #     g = self.df_train.groupby(self.class_column)
        #     df_train = g.apply(lambda x: x.sample(g.size().min())).reset_index(drop=True).sample(frac=1).reset_index(drop=True)
        #     self.train_ds = monai.data.Dataset(data=TTDatasetBX(df_train, mount_point=self.mount_point, img_column=self.img_column, class_column=self.class_column, transform=self.train_transform))

        return DataLoader(self.train_ds, batch_size=self.batch_size, num_workers=self.num_workers, pin_memory=True, drop_last=self.drop_last, collate_fn=self.custom_collate_fn, shuffle=True, prefetch_factor=2)

    def val_dataloader(self):
        # remove balancing for evaluation step for acc or p,r,f metrics
        return DataLoader(self.val_ds, batch_size=self.batch_size, num_workers=self.num_workers, drop_last=self.drop_last, collate_fn=self.custom_collate_fn)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, num_workers=self.num_workers, drop_last=self.drop_last, collate_fn=self.custom_collate_fn)

    def custom_collate_fn(self,batch):
        targets = []
        imgs = []
        for targets_dic in batch:
            img = targets_dic.pop('img', None)
            imgs.append(img.unsqueeze(0))
            targets.append(targets_dic)
        return torch.cat(imgs), targets
    
    def balance_batch_collate_fn(self, batch):
        """Custom collate function that balances classes in a batch"""
        images = torch.stack([item['img'] for item in batch])
       
        masks = torch.stack([item['mask'] for item in batch])

        original_boxes = [item['boxes'] for item in batch]
        original_labels = [item['labels'] for item in batch]
        
        # Count boxes per class across the entire batch
        all_labels = []
        for item in batch:
            all_labels.append(item['labels'])
        
        # Find minimum count across classes
        all_labels = torch.cat(all_labels)
        classes, counts = torch.unique(all_labels, return_counts=True)
        min_count = counts.min().item()
        
        targets = []
        for i, (boxes, labels, mask) in enumerate(zip(original_boxes, original_labels, masks)):
            
            img_classes = torch.unique(labels)
            img_balanced_boxes, img_balanced_labels = [], []

            for cls in img_classes:
                m_label = (labels == cls)
                cls_boxes = boxes[m_label]
                cls_labels = labels[m_label]
                
                # Calculate how many to keep of this class
                proportion = len(cls_boxes) / counts[classes == cls].item()
                keep_count = max(1, int(min_count * proportion))
                keep_count = min(keep_count, len(cls_boxes))
                
                # Randomly sample
                if len(cls_boxes) > keep_count:
                    indices = torch.randperm(len(cls_boxes))[:keep_count]
                    cls_boxes = cls_boxes[indices]
                    cls_labels = cls_labels[indices]
                
                img_balanced_boxes.append(cls_boxes)
                img_balanced_labels.append(cls_labels)
            
            if img_balanced_boxes:
                img_boxes = torch.cat(img_balanced_boxes)
                img_labels = torch.cat(img_balanced_labels)
            else:
                img_boxes = boxes
                img_labels = labels
            
            dic_i = {'labels': img_labels, 
                     'boxes': img_boxes,
                     'mask': mask}
            targets.append(dic_i)

        labels = [t['labels'] for t in targets]
        classes, counts = torch.unique(torch.cat(labels), return_counts=True)
        # print(counts)

        return images, targets
    

def _square_pad_image(img, **kwargs):
    h, w = img.shape[:2]
    s = max(h, w)
    pad = A.PadIfNeeded(min_height=s, min_width=s, border_mode=0)  # constant
    return pad(image=img)["image"]

def _square_pad_mask(msk, **kwargs):
    h, w = msk.shape[:2]
    s = max(h, w)
    pad = A.PadIfNeeded(min_height=s, min_width=s, border_mode=0)  # constant
    return pad(image=msk)["image"]

    

class BBXImageTrainTransform():
    def __init__(self, height=384, width=768):
        self.h = height
        self.w = width

        self.transform = A.Compose(
            [
                # A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                # A.LongestMaxSize(max_size_hw=(self.h, None)),
                A.Lambda(image=_square_pad_image, mask=_square_pad_mask),
                A.Resize(height=1024, width=1024),
                A.Rotate(p=0.5),
                # A.CenterCrop(height=self.h, width=self.w, pad_if_needed=True),
                # A.VerticalFlip(p=0.5),
                A.GaussNoise(),
                A.OneOf(
                    [
                        A.MotionBlur(p=.2),
                        A.MedianBlur(blur_limit=3, p=0.1),
                        A.Blur(blur_limit=3, p=0.1),
                    ], p=0.1),
                A.OneOf(
                    [
                        A.OpticalDistortion(p=0.3),
                        A.GridDistortion(p=.1),
                        ], p=0.1),
                A.OneOf(
                    [
                        A.CLAHE(clip_limit=2),
                        A.RandomBrightnessContrast(),
                    ], p=0.1),
                A.HueSaturationValue(p=0.1),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=.1),
            ], 
            bbox_params=A.BboxParams(format='pascal_voc', min_area=32, min_visibility=0.1, label_fields=['category_ids']),
            additional_targets={'mask': 'mask'},
        )

    def __call__(self, image, bboxes, category_ids, mask):
        return self.transform(image=image, bboxes=bboxes, category_ids=category_ids, mask=mask)

class BBXImageEvalTransform():
    def __init__(self, height=384, width=768):
        self.h = height
        self.w = width

        self.transform = A.Compose(
            [
                # A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                # A.LongestMaxSize(max_size_hw=(self.h, None)),
                # A.VerticalFlip(p=0.5),
                # A.CenterCrop(height=self.h, width=self.w, pad_if_needed=True),
                A.Lambda(image=_square_pad_image, mask=_square_pad_mask),
                A.Resize(height=1024, width=1024),
            ], 
            bbox_params=A.BboxParams(format='pascal_voc', min_area=32, min_visibility=0.1, label_fields=['category_ids']),
            additional_targets={'mask': 'mask'},
        )

    def __call__(self, image, bboxes, category_ids, mask=None):
        return self.transform(image=image, bboxes=bboxes, category_ids=category_ids, mask=mask)
    

class BBXImageTestTransform():
    def __init__(self, height=384, width=768):
        self.h = height
        self.w = width

        self.transform = A.Compose(
            [
                A.NoOp(),                
            ], 
            bbox_params=A.BboxParams(format='pascal_voc', min_area=32, min_visibility=0.1, label_fields=['category_ids']),
            additional_targets={'mask': 'mask'},
        )

    def __call__(self, image, bboxes, category_ids, mask=None):
        return self.transform(image=image, bboxes=bboxes, category_ids=category_ids, mask=mask)

In [ ]:
def plot_batch_with_boxes(
    imgs,
    targets,
    label_names=None,
    max_images=4,
    score_key=None,
    denorm=None,
):
    """
    Plot a batch of images with bounding boxes colored by label.

    Args:
        imgs: Tensor/ndarray/list. Shapes accepted:
              - torch tensor [B,C,H,W] or [C,H,W]
              - numpy [B,H,W,C] or [H,W,C] or [H,W]
              - list of per-image tensors/arrays
        targets: list[dict] length B. Each dict has keys like:
                 - 'boxes' : [N,4] in (x1,y1,x2,y2) image pixel coords
                 - 'labels': [N] int in [0..5]
                 - (optional) 'scores': [N] if you want to filter by score_key
        label_names: optional dict/int->str, e.g. {0:"bg",1:"tool",...}
        max_images: how many images from the batch to display
        score_key: e.g. 'scores' to filter low-confidence boxes if present
        denorm: optional callable(img_tensor)->img_tensor for undoing normalization
    """
    # fixed, distinct colors for labels 0..5
    label_colors = {
        0: "tab:blue",
        1: "tab:orange",
        2: "tab:green",
        3: "tab:red",
        4: "tab:purple",
        5: "tab:brown",
    }

    def to_numpy_img(x):
        # torch -> numpy
        if hasattr(x, "detach"):
            x = x.detach().cpu()
            if denorm is not None:
                x = denorm(x)
            x = x.float()
            x = x.numpy()

        # CHW -> HWC
        if x.ndim == 3 and x.shape[0] in (1, 3):  # CHW
            x = np.transpose(x, (1, 2, 0))
        # HWC ok; HW ok
        return x

    # normalize imgs into a list of per-image arrays/tensors
    if isinstance(imgs, (list, tuple)):
        img_list = list(imgs)
    else:
        # torch tensor or numpy array
        if hasattr(imgs, "ndim") and imgs.ndim == 4:
            img_list = [imgs[i] for i in range(imgs.shape[0])]
        else:
            img_list = [imgs]

    B = min(len(img_list), len(targets), max_images)
    if B <= 0:
        return

    fig, axes = plt.subplots(1, B, figsize=(6 * B, 6))
    axes = [axes] if B == 1 else list(axes)

    for i in range(B):
        img_np = to_numpy_img(img_list[i])
        ax = axes[i]

        # display image
        if img_np.ndim == 2:
            ax.imshow(img_np, cmap="gray")
        else:
            img_disp = img_np
            if img_disp.dtype != np.uint8:
                # best-effort scaling for float images
                vmin = np.nanmin(img_disp)
                vmax = np.nanmax(img_disp)
                if vmax > vmin:
                    img_disp = (img_disp - vmin) / (vmax - vmin)
            ax.imshow(img_disp)

        t = targets[i]
        boxes = t.get("boxes", None)
        labels = t.get("labels", None)

        if boxes is None or labels is None:
            ax.set_title(f"image {i} (no boxes/labels)")
            ax.axis("off")
            continue

        # torch -> numpy
        if hasattr(boxes, "detach"):
            boxes = boxes.detach().cpu().numpy()
        if hasattr(labels, "detach"):
            labels = labels.detach().cpu().numpy()

        # optional score filtering
        if score_key is not None and score_key in t:
            scores = t[score_key]
            if hasattr(scores, "detach"):
                scores = scores.detach().cpu().numpy()
            keep = scores > 0.0
            boxes = boxes[keep]
            labels = labels[keep]

        for (x1, y1, x2, y2), lab in zip(boxes, labels):
            lab_int = int(lab)
            color = label_colors.get(lab_int, "white")
            rect = Rectangle(
                (x1, y1),
                (x2 - x1),
                (y2 - y1),
                fill=False,
                linewidth=2,
                edgecolor=color,
            )
            ax.add_patch(rect)

            name = label_names.get(lab_int, str(lab_int)) if isinstance(label_names, dict) else str(lab_int)
            ax.text(
                x1,
                max(0, y1 - 3),
                name,
                fontsize=10,
                color="white",
                bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor="none"),
            )

        ax.set_title(f"image {i}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()



In [ ]:

def _clamp_xyxy(boxes: torch.Tensor, H: int, W: int) -> torch.Tensor:
    boxes = boxes.clone()
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W - 1)  # x1,x2
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H - 1)  # y1,y2
    return boxes

@torch.no_grad()
def labelmap_from_boxes_intersect_label3(
    imgs: torch.Tensor,
    targets: list[dict],
    mask_value: int = 3,          # intersect with mask == 3
    background: int = 0,
    overlap_rule: str = "last", # "last" | "largest" | "score",
    pad_y = 100,
    pad_x = 5,
) -> torch.Tensor:
    """
    Returns:
      out: LongTensor [B,H,W]
    """
    assert imgs.dim() == 4, f"imgs should be [B,C,H,W], got {imgs.shape}"
    B, _, H, W = imgs.shape
    device = imgs.device

    out = torch.full((B, H, W), background, dtype=torch.long, device=device)

    for b in range(B):
        t = targets[b]
        boxes  = t["boxes"].to(device)
        labels = t["labels"].to(device).long() + 1
        mask_lm = t["mask"].to(device)

        if mask_lm.dim() != 2:
            raise ValueError(f"Expected mask labelmap [H,W], got {mask_lm.shape}")
        if mask_lm.shape != (H, W):
            raise ValueError(f"mask shape {mask_lm.shape} must match image {(H,W)}")

        if boxes.numel() == 0:
            continue

        # boolean region of interest: only where labelmap == mask_value (i.e., == 3)
        roi = (mask_lm == mask_value)        

        # clamp + integerize boxes, use exclusive slicing
        boxes = _clamp_xyxy(boxes, H, W)
        x1 = boxes[:, 0].floor().long().clamp(0, W - 1)
        y1 = boxes[:, 1].floor().long().clamp(0, H - 1)
        x2 = boxes[:, 2].ceil().long()
        y2 = boxes[:, 3].ceil().long()

        # make x2/y2 exclusive and at least 1 pixel wide/tall
        x2 = torch.maximum(x2, x1 + 1).clamp(1, W)
        y2 = torch.maximum(y2, y1 + 1).clamp(1, H)

        N = boxes.shape[0]

        # choose painting order (later overwrites earlier)
        if overlap_rule == "last":
            order = torch.sort(labels)[1]
            # order = torch.arange(N, device=device)
        elif overlap_rule == "largest":
            areas = (x2 - x1) * (y2 - y1)
            order = torch.argsort(areas)  # small -> large; large painted last = wins
        elif overlap_rule == "score":
            if "scores" not in t:
                raise ValueError("overlap_rule='score' requires targets[i]['scores']")
            scores = t["scores"].to(device)
            order = torch.argsort(scores)  # low -> high; high painted last = wins
        else:
            raise ValueError(f"Unknown overlap_rule: {overlap_rule}")

        lm = roi.clone().long() 

        for j in order.tolist():
            xa, xb = int(x1[j]) - pad_x, int(x2[j]) + pad_x
            ya, yb = int(y1[j]) - pad_y, int(y2[j]) + pad_y

            xa = max(xa, 0)
            xb = min(xb, W)
            ya = max(ya, 0)
            yb = min(yb, H)
            if xa >= xb or ya >= yb:
                continue

            allowed = roi[ya:yb, xa:xb]
            if allowed.any():
                patch = lm[ya:yb, xa:xb]
                patch[allowed] = labels[j]
                lm[ya:yb, xa:xb] = patch

        out[b] = lm

    return out


In [ ]:
# model_fn = '/MEDUSA_STOR/jprieto/EGower/train_output/segmentation/ttunet/full_seg/v5.3/epoch=189-val_loss=0.17.pt'
# model = torch.jit.load(model_fn)
# model.eval()
# model.cuda()

In [ ]:
mount_point = '/MEDUSA_STOR/jprieto/EGower/'
csv_train = os.path.join(mount_point, 'CSV_files/mtss_pret_combined_train_train.csv')
csv_val = os.path.join(mount_point, 'CSV_files/mtss_pret_combined_train_test.csv')
csv_test = os.path.join(mount_point, 'CSV_files/mtss_pret_combined_test.csv')


In [ ]:

dm = TTDataModuleBX(
    df_train = pd.read_csv(csv_train),
    df_val = pd.read_csv(csv_val),
    df_test = pd.read_csv(csv_test),
    mount_point=mount_point,
    batch_size=1,
    num_workers=1,
    img_column="filename",
    class_column='class',
    balanced=False,
    train_transform=BBXImageTestTransform(768,1536), 
    valid_transform=BBXImageEvalTransform(768,1536), 
    test_transform=BBXImageTestTransform(768,1536),
    drop_last=False
)
dm.setup()

In [ ]:
train_dl = dm.train_dataloader()

In [ ]:
batch = next(iter(train_dl))
imgs, targets = batch

In [ ]:
plot_batch_with_boxes(
    imgs,
    targets,
    max_images=2,
)

In [ ]:
df_train = pd.read_csv(csv_train)

In [ ]:
df_train.columns

In [ ]:
df_train[['label', 'class']].drop_duplicates().sort_values('class')

In [ ]:
labelmaps = labelmap_from_boxes_intersect_label3(
    imgs=imgs,
    targets=targets,
    background=0,
    overlap_rule="last",
)

In [ ]:
label_colors = {    
    0: "black",
    1: "tab:blue",
    2: "tab:orange",
    3: "tab:green",
    4: "tab:red",
    5: "tab:purple",
    6: "tab:brown",
}

names = {1:"healthy", 2:"ECA", 3:"Entropion", 4:"Gap", 5:"Fleshy", 6:"overcorrection"}


plot_image(imgs, labelmaps, idx=0, alpha=0.6, label_colors=label_colors, background_label=0, colorbar_labels=names)


In [ ]:
batch = pickle.load(open('/MEDUSA_STOR/jprieto/EGower/mtss_pred_combined_merged/PoPP_Data/mtss/img/11309_12Intra_od_postop.pkl', 'rb'))

In [ ]:
plot_image(batch['img'], batch['seg'], alpha=0.6, label_colors=label_colors, background_label=0, colorbar_labels=names)

In [ ]:
temp_img = sitk.GetArrayFromImage(sitk.ReadImage('/MEDUSA_STOR/jprieto/EGower/B images one eye/img/2220_B-RIGHT.jpg'))
temp_seg = sitk.GetArrayFromImage(sitk.ReadImage('/MEDUSA_STOR/jprieto/EGower/B images one eye/seg/2220_B-RIGHT.nrrd'))

plot_image(torch.tensor(temp_img).permute(2,0,1).unsqueeze(0)/255.0, torch.tensor(temp_seg).unsqueeze(0), idx=0, alpha=0.6, label_colors=label_colors, background_label=0, colorbar_labels=names)

In [ ]:
test_dl = dm.test_dataloader()
batch = next(iter(test_dl))
img, targets = batch


In [ ]:
lm = labelmap_from_boxes_intersect_label3(
    imgs=img,
    targets=targets,
    background=0,
    overlap_rule="last",
)

In [ ]:
plot_image(img, lm, alpha=0.6, label_colors=label_colors, background_label=0, colorbar_labels=names)

In [ ]:
import pickle
img_seg = pickle.load(open('/MEDUSA_STOR/jprieto/EGower/mtss_pred_combined_merged_segpredv53/B images one eye/img/1380_B-LEFT.pkl', 'rb'))

In [ ]:
img = img_seg['img']
seg = img_seg['seg']

plot_image(img, seg, alpha=0.6, label_colors=label_colors, background_label=0, colorbar_labels=names)